# DDoS Auswertung

## 0. Foreplay

### Notes:

instead of polars, the polars-lts-cpu package is used, to run on the old CPU's of the mobi8

### Imports

In [13]:
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pyarrow.parquet as pq

### Global parameters

In [2]:
SERVER_IP = "141.22.28.227"

### external IANA protocol list to convert numbers to names

In [3]:
# read IANA list to convert number to text
lf_pro_nb = (
    pl.scan_csv("data/external/protocol-numbers-1.csv", ignore_errors=True)
      .select("Decimal", "Keyword")
).rename({"Keyword": "ip.proto.name", "Decimal": "ip.proto.num"})
lf_pro_nb.head(10).collect()

ip.proto.num,ip.proto.name
i64,str
0,"""HOPOPT"""
1,"""ICMP"""
2,"""IGMP"""
3,"""GGP"""
4,"""IPv4"""
5,"""ST"""
6,"""TCP"""
7,"""CBT"""
8,"""EGP"""


In [4]:
# lf = (
#     lf.join(lf_pro_nb, left_on="ip.proto", right_on="ip.proto.num", how="left")
#     .rename({"ip.proto": "ip.proto.num"})
#     .pipe(lambda lf: lf.select(
#     (cols := lf.collect_schema().names(), cols.insert(5, cols.pop(cols.index("ip.proto.name"))), cols)[-1]
# ))
# )

# lf.head(10).collect()

### Read

In [5]:
lf = pl.scan_parquet("data/interim/ddos_1/batch_1_1000.parquet").drop("dtls.record.content_type", "dtls.record.version", "dtls.handshake.type")

## 1. General Analysis

### Overview

total_packets = lf.select(pl.len()).collect().item()
total_bytes = lf.select(pl.col("frame.len").sum()).collect().item()

t_start = lf.select(pl.col("frame.datetime").min()).collect().item()
t_end = lf.select(pl.col("frame.datetime").max()).collect().item()
if t_start and t_end != None:
    duration_s = (t_end - t_start).total_seconds()

avg_len = (
    lf.select(pl.col("frame.len").mean())
    .collect()
    .item()
)
max_len = (
    lf.select(pl.col("frame.len").max())
    .collect()
    .item()
)

top_protocols = (
    lf.group_by("frame.protocols")
      .len()
      .sort("len", descending=True)
      .limit(10)
      .collect()
)

print(f"All pakets          : {total_packets:>12}")
print(f"All bytes           : {total_bytes:>12,}  ({total_bytes/1e6:.1f} MB)")
print(f"Timepoints          : {t_start}  →  {t_end}")
print(f"Duration            : {duration_s:.1f}s / {duration_s/60:.1f}min / {duration_s/60/60:.1f}hours")
print(f"Ø Packetrate        : {total_packets/duration_s:>10,.1f} pkt/s")
print(f"Ø Bitrate           : {total_bytes*8/duration_s/1e6:>10,.1f} Mbit/s")
print(f"Average packet size : {avg_len:.1f} bytes")
print(f"Maximum packet size : {max_len} bytes")
print(top_protocols)

### Sample

In [6]:
df_sample = lf.limit(1000).collect().sample(10)
df_sample

frame.datetime,frame.len,frame.protocols,ip.src,ip.dst,ip.proto,ip.ttl,ipv6.src,ipv6.dst,ipv6.hlim,udp.srcport,udp.dstport,udp.length,tcp.srcport,tcp.dstport,tcp.len,coap.version,coap.type,coap.code,coap.mid,coap.token,coap.token_len,coap.opt.uri_path,coap.opt.uri_query,coap.opt.block_number,coap.opt.block_mflag,coap.opt.block_size,coap.opt.observe,coap.response_to,coap.response_in,coap.payload_length,source_file
datetime[μs],i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2025-08-13 15:51:49.578367,1514,"""eth:ethertype:ip:data""","""150.249.242.21""","""141.22.28.227""","""17""","""49""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.576972,899,"""eth:ethertype:ip:data""","""125.35.65.210""","""141.22.28.227""","""17""","""47""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.579223,1514,"""eth:ethertype:ip:udp:cldap""","""217.154.80.170""","""141.22.28.227""","""17""","""119""",null,null,null,"""389""","""28547""",2977,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.703993,482,"""eth:ethertype:ip:udp:ntp""","""80.95.136.74""","""141.22.28.227""","""17""","""53""",null,null,null,"""123""","""46288""",448,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574188,1514,"""eth:ethertype:ip:udp:dns""","""38.52.217.173""","""141.22.28.227""","""17""","""52""",null,null,null,"""53""","""60225""",3732,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.577727,1514,"""eth:ethertype:ip:udp:data""","""86.20.238.96""","""141.22.28.227""","""17""","""50""",null,null,null,"""3702""","""13140""",2360,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.705080,256,"""eth:ethertype:ip:udp:data""","""200.3.220.53""","""141.22.28.227""","""17""","""120""",null,null,null,"""1434""","""6575""",222,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.704753,1514,"""eth:ethertype:ip:udp:coap""","""111.38.47.50""","""141.22.28.227""","""17""","""49""",null,null,null,"""5683""","""19554""",1524,null,null,null,"""1""","""2""","""69""","""462""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.575576,1514,"""eth:ethertype:ip:data""","""61.221.40.210""","""141.22.28.227""","""17""","""48""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"


In [7]:
double_ip = lf.filter(
    pl.col("ip.ttl").str.contains(",") | pl.col("ip.proto").str.contains(",")
).collect()
print(double_ip.shape)
double_ip

(6821217, 32)


frame.datetime,frame.len,frame.protocols,ip.src,ip.dst,ip.proto,ip.ttl,ipv6.src,ipv6.dst,ipv6.hlim,udp.srcport,udp.dstport,udp.length,tcp.srcport,tcp.dstport,tcp.len,coap.version,coap.type,coap.code,coap.mid,coap.token,coap.token_len,coap.opt.uri_path,coap.opt.uri_query,coap.opt.block_number,coap.opt.block_mflag,coap.opt.block_size,coap.opt.observe,coap.response_to,coap.response_in,coap.payload_length,source_file
datetime[μs],i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2025-08-13 15:51:49.574617,70,"""eth:ethertype:ip:icmp:ip:udp""","""186.9.151.108,141.22.28.227""","""141.22.28.227,186.9.151.108""","""1,17""","""47,240""",null,null,null,"""41505""","""3283""",12,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.707449,164,"""eth:ethertype:ip:icmp:ip:udp:s…","""213.229.241.118,141.22.28.227""","""141.22.28.227,213.229.241.118""","""1,17""","""53,234""",null,null,null,"""20595""","""1900""",102,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.707877,164,"""eth:ethertype:ip:icmp:ip:udp:s…","""105.106.72.193,141.22.28.227""","""141.22.28.227,105.106.72.193""","""1,17""","""52,241""",null,null,null,"""50552""","""1900""",102,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.712003,74,"""eth:ethertype:ip:icmp:ip:udp:d…","""213.161.189.75,141.22.28.227""","""141.22.28.227,213.161.189.75""","""1,17""","""36,225""",null,null,null,"""45236""","""3283""",12,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.718220,74,"""eth:ethertype:ip:icmp:ip:udp:d…","""1.29.78.183,141.22.28.227""","""141.22.28.227,1.29.78.183""","""1,17""","""109,239""",null,null,null,"""35155""","""3283""",12,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-13 17:46:30.451268,557,"""eth:ethertype:ip:icmp:ip:udp:d…","""141.22.28.227,171.96.99.141""","""171.96.99.141,141.22.28.227""","""1,17""","""64,46""",null,null,null,"""41486""","""53386""",495,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 17:46:30.451270,510,"""eth:ethertype:ip:icmp:ip:udp:n…","""141.22.28.227,183.80.175.24""","""183.80.175.24,141.22.28.227""","""1,17""","""64,45""",null,null,null,"""123""","""62462""",448,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 17:46:30.458506,70,"""eth:ethertype:ip:icmp:ip:udp""","""220.201.110.193,141.22.28.227""","""141.22.28.227,220.201.110.193""","""1,17""","""46,240""",null,null,null,"""62735""","""5683""",29,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"


### Datenübersicht

#### Incoming vs. Outgoing

In [8]:
def split_direction(lf: pl.LazyFrame, server_ip):
    lf_out = lf.filter(pl.col("ip.src") == server_ip)
    lf_in = lf.filter(pl.col("ip.dst") == server_ip)
    lf_null = lf.filter(
        (pl.col("ip.dst") != server_ip) & (pl.col("ip.src") != server_ip)
    )
    lf_null = lf.filter(
        pl.col("ip.src").is_null() | pl.col("ip.dst").is_null()
    )
    
    return lf_out, lf_in, lf_null

In [9]:
lf_out, lf_in, lf_null = split_direction(lf, SERVER_IP)

size = lf.select(pl.len()).collect().item()
size_out = lf_out.select(pl.len()).collect().item()
size_in = lf_in.select(pl.len()).collect().item()
size_null = lf_null.select(pl.len()).collect().item()

print(f"Packets gesamt:{size:3,}\n")
print(f"Anteil out: {size_out/size:.6}%")
print(f"Packets out:", size_out)
print(f"Anteil in: {size_in/size:.2}%")
print(f"Packets in:", size_in)
print(f"Anteil null: {size_null / size:.6f}%")
print(f"Packets null:", size_null)

Packets gesamt:696,340,178

Anteil out: 8.03774e-06%
Packets out: 5597
Anteil in: 0.99%
Packets in: 689498675
Anteil null: 0.000020%
Packets null: 13965


#### By Transport Protocol

In [ ]:
TOP_N = 5

lf_by_proto = (
    lf.group_by("ip.proto")
    .agg(pl.len().alias("count"))
    .collect()
    .sort("count", descending=True)
)

total = lf_by_proto["count"].sum()

top = lf_by_proto.head(TOP_N)
other = lf_by_proto.tail(len(lf_by_proto) - TOP_N)
other_count = other["count"].sum() if len(other) > 0 else 0

labels = top["ip.proto"].to_list()
counts = top["count"].to_list()

if other_count > 0:
    labels.append("Other")
    counts.append(other_count)

percentages = [c / total * 100 for c in counts]
colors = cm.tab10.colors[: len(labels)]

fig, ax = plt.subplots(figsize=(10, 3))

left = 0
for label, count, pct, color in zip(labels, counts, percentages, colors):
    ax.barh(0, count, left=left, color=color, edgecolor="white", label=label)
    ax.text(
        left + count / 2,
        0,
        f"{pct:.2f}%",
        ha="center",
        va="center",
        fontsize=9,
        color="white" if pct > 3 else "black",
    )
    left += count

ax.set_xlim(0, total)
ax.set_yticks([])
ax.set_xlabel("Packets")
ax.set_title("IP Protocol Distribution")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=len(labels))

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))

y_pos = range(len(labels))
ax.barh(y_pos, counts, color=colors)

for i, (count, pct) in enumerate(zip(counts, percentages)):
    ax.text(count * 1.05, i, f"{pct:.2f}%", va="center", fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xscale("log")
ax.set_xlabel("Packets (log scale)")
ax.set_title("IP Protocol Distribution")

plt.tight_layout()
plt.show()

NameError: name 'other_count' is not defined

#### Top-Talker (meiste Pakete / meiste Bytes)

In [11]:
talkers = (
    lf.group_by("ip.src")
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("bytes", descending=True)
)
top_talkers = talkers.head(20).collect()
top_talkers

ip.src,packets,bytes
str,u32,i64
"""163.181.207.4""",2090918,1007822476
"""163.181.207.2""",2087317,1006086794
"""163.181.207.1""",2087046,1005956172
"""163.181.207.3""",1960720,945067040
"""195.69.189.18""",1888460,910237720
…,…,…
"""201.62.49.58""",696153,335545746
"""217.73.144.39""",675246,325468572
"""124.120.113.203""",662477,319313914


In [12]:
lf_ipv6 = lf.filter(
    pl.col("ipv6.src").is_not_null() | pl.col("ipv6.dst").is_not_null()
).collect()
lf_ipv6

frame.datetime,frame.len,frame.protocols,ip.src,ip.dst,ip.proto,ip.ttl,ipv6.src,ipv6.dst,ipv6.hlim,udp.srcport,udp.dstport,udp.length,tcp.srcport,tcp.dstport,tcp.len,coap.version,coap.type,coap.code,coap.mid,coap.token,coap.token_len,coap.opt.uri_path,coap.opt.uri_query,coap.opt.block_number,coap.opt.block_mflag,coap.opt.block_size,coap.opt.observe,coap.response_to,coap.response_in,coap.payload_length,source_file
datetime[μs],i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2025-08-13 15:51:51.835864,1374,"""eth:ethertype:ip:udp:teredo:ip…","""106.8.207.199""","""141.22.28.227""","""17""","""51""","""2f3e:3b74:6974:6c65:3d22:4765:…","""6c20:496e:666f:223b:6374:3d30:…","""60""","""5683""","""3544""",1340,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:52.796131,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe92:6664""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:57.292536,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe99:e4cc""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:57.478403,162,"""eth:ethertype:ip:udp:amt:ip:ip…","""181.199.119.134""","""141.22.28.227""","""17""","""118""","""3b49:6e73:7461:6e63:654e:616d:…","""5351:4c53:4552:5645:523b:4973:…","""82""","""1434""","""2268""",128,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:58.479987,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe97:c8f7""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-13 17:46:26.323593,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe96:ee87""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 17:46:27.700537,165,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe96:2755""","""ff02::1:2""","""1""","""546""","""547""",107,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 17:46:28.782771,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe96:c58c""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"


In [ ]:
protocol_counts = (
    lf.group_by("frame.protocols")
      .len()
      .sort("len", descending=True)
      .head(5)
      .collect()
)

plt.figure(figsize=(8, 8))
plt.pie(
    protocol_counts["len"],
    labels=protocol_counts["frame.protocols"].to_list(),
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Verteilung der Protokolle")
plt.axis("equal")  # Kreis statt Ellipse
plt.show()